# Reproducible model comparison

Both models exclude duration. The 70/15/15 stratified split uses seed 42. Candidate thresholds maximize validation F1; validation average precision selects the model. Only that model is evaluated on the held-out test.


In [1]:
from pathlib import Path
import os
ROOT = Path.cwd() if (Path.cwd() / "src/train.py").exists() else Path.cwd().parent
os.chdir(ROOT)


In [2]:
from src.train import train
results = train()


{
  "data_file": "bank-additional-full.csv",
  "data_sha256": "74adfc578bf77a7ff4bb1ba4a9f8709d9e3c6907342959c2c8416847e0afb4d8",
  "split": {
    "train": 28831,
    "validation": 6178,
    "test": 6179,
    "random_state": 42,
    "stratified": true
  },
  "excluded_features": [
    "duration"
  ],
  "selection_metric": "validation average precision (labelled PR-AUC for compatibility)",
  "threshold_selection": "maximum validation F1 for each candidate",
  "validation": {
    "Logistic Regression": {
      "ROC-AUC": 0.8021599948839883,
      "PR-AUC": 0.4700697725957582,
      "Precision": 0.46365914786967416,
      "Recall": 0.5316091954022989,
      "F1": 0.49531459170013387,
      "threshold": 0.2207722911559955
    },
    "Random Forest": {
      "ROC-AUC": 0.7828242482188311,
      "PR-AUC": 0.4406762080472387,
      "Precision": 0.44143033292231815,
      "Recall": 0.514367816091954,
      "F1": 0.4751161247511613,
      "threshold": 0.28
    }
  },
  "selected_model": "Logist

In [3]:
import pandas as pd
pd.DataFrame(results["validation"]).T


,ROC-AUC,PR-AUC,Precision,Recall,F1,threshold
Logistic Regression,0.802160,0.470070,0.463659,0.531609,0.495315,0.220772
Random Forest,0.782824,0.440676,0.441430,0.514368,0.475116,0.280000


In [4]:
pd.Series(results["test"], name=results["selected_model"])


ROC-AUC      0.804743
PR-AUC       0.468097
Precision    0.467066
Recall       0.560345
F1           0.509471
Name: Logistic Regression, dtype: float64

In [5]:
import joblib
artifact = joblib.load("models/model.joblib")
assert artifact["threshold"] == results["threshold"]
assert "duration" not in artifact["feature_columns"]
print("Saved model and threshold:", artifact["threshold"])


Saved model and threshold: 0.2207722911559955


## Interpretation

The quantity labelled PR-AUC in the original code is sklearn average_precision_score (AP), not trapezoidal area. The random split is a reproducible baseline; this date-ordered dataset also needs a chronological evaluation before deployment claims. Unknown values are retained as explicit categorical levels. This notebook does not claim production readiness.
